<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/2-2_chromadb-server-mode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1 style="text-align: center;">Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>2.2 ChromaDB Server Mode</h2>
    <p>by <b>Pere Martra</b></p>
</div>

_________
This notebook is set up to run in a local environment. If executed in Google Colab, access to the ChromaDB server that will be created won't be possible.

The code is the same as the one used in notebook 2-1_vector-databases-llms.ipynb. 
The client has been implemented in the notebook 2-3_chromadb-client. 
__________

In [ ]:
%pip install chromadb

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# The dataset is mirrored on GitHub, so no Kaggle account or token is needed.
# We download it once to a local CSV (identical behaviour on Colab and locally).
import os
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/kotartemiy/topic-labeled-news-dataset/master/labeled_newscatcher_dataset.csv"
csv_path = "labeled_newscatcher_dataset.csv"
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(DATA_URL, csv_path)

news = pd.read_csv(csv_path, sep=';')
MAX_NEWS = 1000
DOCUMENT = "title"
TOPIC = "topic"

In [ ]:
#Because it is just a example we select a small portion of News.
subset_news = news.head(MAX_NEWS)

In [ ]:
import chromadb

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chromadb")

In [ ]:
collection = chroma_client.get_or_create_collection(name="local_news_collection")

In [ ]:
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

In [ ]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

In [ ]:
#Running Chroma in Server Mode
!chroma run --path ./chromadb

You have the code to test this server in the notebook 2-3_chromadb-client.ipynb